# 04b - Ridership (leak-free monthly feature)

Builds `avg_daily_boards_monthly`, a leak-free per (line, year, month) ridership feature that `07` merges in. 

Uses `04_ridership.ipynb`'s annual base and fallback index (`4_ridership_annual_by_line.parquet`, `4_ridership_monthly_index_fallback.parquet`) plus a fresh fetch of the system-wide monthly dataset (which is small/fast) for trailing index/YTD ratios.

In [1]:
import pandas as pd
import numpy as np

from utils import line_key, fetch_ridership_monthly

BASEPATH = "../data"

## Fetch system-wide monthly ridership

Same public dataset `04_ridership.ipynb` reads. Refetched directly here rather than sharing since the fetch itself is small/fast and this keeps this notebook independent of a recent run of `04`.

In [2]:
rr_monthly_raw = fetch_ridership_monthly()

# trailing 12-month mean ending at (and including) each row 
# min_periods = 1 lets the partial window before 12 months of 
# history exist (2017) still produce a value.
# same pattern as this project's other lag features
rr = rr_monthly_raw.copy()
rr["trailing_mean"] = rr["ridership"].rolling(window = 12, min_periods = 1).mean()
rr["trailing_index_safe"] = rr["ridership"] / rr["trailing_mean"]
rr = rr[["ref_year", "ref_month", "trailing_index_safe"]]

print(f"trailing index: {len(rr)} rows")
rr.tail()

trailing index: 87 rows


,ref_year,ref_month,trailing_index_safe
82,2025,11,0.949209
83,2025,12,0.929634
84,2026,1,0.906498
85,2026,2,0.906396
86,2026,3,0.980794


## Leak-free ridership feature

`avg_daily_boards_monthly` normalization risks leaking of future months' totals into past rows w/o being careful. This section explains what value gets used for a given `(line, year, month)` row in the mapping table

- Rows anchored to the last completed month: Every row is joined to the last full calendar month before its own scheduled departure.
  - Ex: A row scheduled in April 2025 gets ridership figures referencing March 2025, since April itself isn't over yet and a predictive model shouldnt know Aprils total ridership

- Annual base and monthly index, by period: `avg_daily_boards_monthly = annual_base x monthly_index`, using whichever pair below applies to a row's reference month (differences are due to data coverage issues presented in 04 + train/test considerations):

  | Reference month | Annual base | Monthly index |
  |---|---|---|
  | 2017 and 2018 | `04`'s imputed per-line annual figure | `04`'s full-year fallback index (2017/2018 imputed as the average of "normal" years -- see `04`'s seasonality check) |
  | 2019 through Nov 2024 | `04`'s annual figure (real survey for 2017, 2019, 2022-2024 // imputed for 2018, 2020, 2021) | Trailing 12-month index, built below |
  | Dec 2024 through Nov 2025 (this project's 2025 test year) | YTD-ratio-scaled 2024 figure, built below | Trailing 12-month index, built below |

- 
- Trailing (not whole-year) monthly index: `trailing_index[year, month] = that month's ridership / the trailing 12-month mean ending at and including that month` (min_periods = 1 for the partial window before 12 months of history available, in 2017).
  - This only ever depends on data up to/including the (ALREADY LAGGED BY 1) month itself vs. the full-year total we have for most years (from OpenDataPhilly/SEPTA pubs). Using the annual totals as the denominator would include info from months that haven't occured yet (relative to the scheduled run in a given row) so would leak mild info into training

- YTD-ratio-scaled 2025 base: As far as I can tell, SEPTA didn't publish a per-line annual ridership survey anywhere for 2025, the way it did for 2017,19,22,23,24 -- I only have monthly reports. For missing training years (2018, 20, 21), I summed months + used that as annual baseline figure in the avg_daily_boards_monthly = annual_base x monthly_index. I justified because the years have already passed, relative to model awareness - not holding them to a strict per-row leak standard for this one metric. 
- But I can't sum 2025 monthly figures to get annual base here since (like mentioned above) it'll leak 2025's full-year denom to test set. Instead, I carry forward 2024 monthly ridership + scale each line's 2024 figure by how much 2025 ridership has actually run above/below 2024 over the same elapsed months so far. January 2025 rows here get scale_ratio = 1.0.
  - Example: an April-2025 row's reference month is March 2025, so it uses the real Jan-Mar 2025 vs. Jan-Mar 2024. Use that as annual_base for 2025.

- Training data imputation notes:
  - The pre-2019 fallback and 2018/2020/2021's imputed annual base (built in `04`) are both confined to
  training years -- none of this reaches the 2025 test set.
  - Only `2017-01` has no reference at all (nothing on either side of either dataset's coverage): 13 line-months, 20,310 rows (1.3%) left `NaN`.

## Read `04`'s annual base + fallback index

In [3]:
df_rider_line = pd.read_parquet(f"{BASEPATH}/4_ridership_annual_by_line.parquet")
df_rider_line["line_key"] = df_rider_line["line"].map(line_key)

# annual base for 2017-2024
# 04_ doesn't estimate 2025 (test year) at all; built below instead
annual_base = df_rider_line[["line_key", "year", "avg_daily_boards"]].drop_duplicates()

# fallback index for before SEPTA's monthly dataset begins in 2019
fallback_index = pd.read_parquet(f"{BASEPATH}/4_ridership_monthly_index_fallback.parquet").rename(
    columns = {"monthly_index": "trailing_index_fallback"}
)

# YTD-ratio-scaled 2025 base -- see markdown above
def _ytd_cumsum(df, year):
    sub = df[df["ref_year"] == year].sort_values("ref_month").copy()
    sub["cum_ridership"] = sub["ridership"].cumsum()
    return sub.set_index("ref_month")["cum_ridership"]

cum_2024 = _ytd_cumsum(rr_monthly_raw, 2024)
cum_2025 = _ytd_cumsum(rr_monthly_raw, 2025)

# ref = (2024,12) is a January 2025 row: no 2025 data exists yet
# here, ratio = 1.0 (I just take 2024 numbers)
# after Jan, start calculating real YTD ratio compared to 2024
ytd_ratio_table = pd.DataFrame({"ref_month": range(1, 12), "ref_year": 2025})
ytd_ratio_table["ytd_ratio"] = ytd_ratio_table["ref_month"].map(lambda m: cum_2025.loc[m] / cum_2024.loc[m])
ytd_ratio_table = pd.concat(
    [pd.DataFrame({"ref_year": [2024], "ref_month": [12], "ytd_ratio": [1.0]}), ytd_ratio_table],
    ignore_index = True,
)

print("2025 YTD ratio vs. same-period 2024 (system-wide Regional Rail):")
print(ytd_ratio_table)

# scale each line's real 2024 figure -- month-dependent for 2025 only
base_2024 = annual_base[annual_base["year"] == 2024][["line_key", "avg_daily_boards"]].rename(
    columns = {"avg_daily_boards": "avg_daily_boards_2024"}
)
annual_base_2025 = base_2024.merge(ytd_ratio_table, how = "cross")
annual_base_2025["avg_daily_boards_2025_scaled"] = (
    annual_base_2025["avg_daily_boards_2024"] * annual_base_2025["ytd_ratio"]
)
annual_base_2025 = annual_base_2025[["line_key", "ref_year", "ref_month", "avg_daily_boards_2025_scaled"]]

print(f"\nannual_base: {annual_base.shape}, lines: {sorted(annual_base['line_key'].unique())}")

2025 YTD ratio vs. same-period 2024 (system-wide Regional Rail):
    ref_year  ref_month  ytd_ratio
0       2024         12   1.000000
1       2025          1   1.099321
2       2025          2   1.124375
3       2025          3   1.109190
4       2025          4   1.097370
5       2025          5   1.091345
6       2025          6   1.086572
7       2025          7   1.086880
8       2025          8   1.078646
9       2025          9   1.077965
10      2025         10   1.061057
11      2025         11   1.045107

annual_base: (104, 3), lines: ['Airport', 'Chestnut Hill East', 'Chestnut Hill West', 'Cynwyd', 'Fox Chase', 'Lansdale/Doylestown', 'Manayunk/Norristown', 'Media', 'Paoli/Thorndale', 'Trenton', 'Warminster', 'West Trenton', 'Wilmington/Newark']


In [4]:
# every (line, year, month) combo this project needs, 
# with its "last completed month" reference
lines = sorted(annual_base["line_key"].unique())
grid = pd.DataFrame(
    [(l, y, m) for l in lines for y in range(2017, 2026) for m in range(1, 13)],
    columns = ["line_key", "year", "month"],
)
grid["ref_year"] = np.where(grid["month"] == 1, grid["year"] - 1, grid["year"])
grid["ref_month"] = np.where(grid["month"] == 1, 12, grid["month"] - 1)

# prefer the real trailing index; fall back to the whole-year 
# index only where no trailing data exists (ref before 2019) 
# true NaN only where neither exists (ref = 2016-12 for Jan 2017)
grid = grid.merge(rr, on = ["ref_year", "ref_month"], how = "left")
grid = grid.merge(fallback_index, on = ["ref_year", "ref_month"], how = "left")
grid["index_used"] = grid["trailing_index_safe"].fillna(grid["trailing_index_fallback"])
grid["used_fallback"] = grid["trailing_index_safe"].isna() & grid["trailing_index_fallback"].notna()

# annual base: standard (line_key, year) merge covers 2017-2024; 
# 2025 gets no match there (annual_base has no 2025 rows at all), 
# so fill those from the YTD-ratio-scaled table, keyed on
# (line_key, ref_year, ref_month) since that base is month-dependent 
# for 2025 rather than a single flat annual number
grid = grid.merge(annual_base, on = ["line_key", "year"], how = "left")
grid = grid.merge(annual_base_2025, on = ["line_key", "ref_year", "ref_month"], how = "left")
grid["avg_daily_boards"] = grid["avg_daily_boards"].fillna(grid["avg_daily_boards_2025_scaled"])
grid = grid.drop(columns = ["avg_daily_boards_2025_scaled"])

assert grid.loc[grid["year"] == 2025, "avg_daily_boards"].isna().sum() == 0, "2025 base still has gaps"
grid["avg_daily_boards_monthly"] = grid["avg_daily_boards"] * grid["index_used"]

print("used fallback (training-only) index:", grid["used_fallback"].sum(), "line-months")
print("no index at all (true NaN, expect only Jan-2017):", grid["index_used"].isna().sum(), "line-months")
print(grid.loc[grid["index_used"].isna(), ["line_key", "year", "month"]].drop_duplicates()["month"].unique())

used fallback (training-only) index: 312 line-months
no index at all (true NaN, expect only Jan-2017): 13 line-months
[1]


## Save

In [5]:
# keep the column named line_key (not line) -- it's already the collapsed join key (e.g. "Media"),
# not a raw line name, and 07's merge should join on it directly with no further mapping needed
safe = grid[["line_key", "year", "month", "avg_daily_boards_monthly"]]
safe.to_parquet(f"{BASEPATH}/4_ridership_by_line_month_safe.parquet", index = False)
print(f"Saved {len(safe):,} rows to 4_ridership_by_line_month_safe.parquet")

Saved 1,404 rows to 4_ridership_by_line_month_safe.parquet
